#YB 미니 프로젝트3 - YB 1조
##'영화 관객수 예측 경진대회' 데이터셋에 회귀 알고리즘을 적용하기

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import ExtraTreesRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.base import clone


In [ ]:
train=pd.read_csv(r"C:\Users\jk102\OneDrive - 이화여자대학교\바탕 화면\이화여자대학교\ESAA\ESAA YB 미니프로젝트(3)\movies_train.csv")
test=pd.read_csv(r"C:\Users\jk102\OneDrive - 이화여자대학교\바탕 화면\이화여자대학교\ESAA\ESAA YB 미니프로젝트(3)\movies_test.csv")
submission=pd.read_csv(r"C:\Users\jk102\OneDrive - 이화여자대학교\바탕 화면\이화여자대학교\ESAA\ESAA YB 미니프로젝트(3)\submission.csv")

In [ ]:
# ============================
# 1) 날짜 처리
# ============================
train['release_dt'] = pd.to_datetime(train['release_time'])
test['release_dt'] = pd.to_datetime(test['release_time'])

train['year'] = train['release_dt'].dt.year
train['month'] = train['release_dt'].dt.month
train['weekday'] = train['release_dt'].dt.weekday
train['is_weekend'] = train['weekday'].isin([4, 5, 6]).astype(int)

test['year'] = test['release_dt'].dt.year
test['month'] = test['release_dt'].dt.month
test['weekday'] = test['release_dt'].dt.weekday
test['is_weekend'] = test['weekday'].isin([4, 5, 6]).astype(int)

# vacation (기존 정의 유지)
train['vacation'] = train['month'].isin([1, 7, 8, 12]).astype(int)
test['vacation'] = test['month'].isin([1, 7, 8, 12]).astype(int)

# sweet spot (상영시간 110~150분)
train['sweet_spot'] = train['time'].between(110, 150).astype(int)
test['sweet_spot'] = test['time'].between(110, 150).astype(int)

# ============================
# 2) 감독 관련 처리
# ============================
# 감독 신인 여부
train['new_director'] = (train['dir_prev_num'] == 0).astype(int)
test['new_director'] = (test['dir_prev_num'] == 0).astype(int)

# 전작 관객수 로그
train['dir_prev_bfnum_log'] = np.log1p(train['dir_prev_bfnum'])
test['dir_prev_bfnum_log'] = np.log1p(test['dir_prev_bfnum'])

# ============================
# 3) distributor median rank (기존 유지)
# ============================
dist_rank = train.groupby('distributor')['box_off_num'].median().rank()
train['distributor_median_rank'] = train['distributor'].map(dist_rank)
test['distributor_median_rank'] = test['distributor'].map(dist_rank).fillna(dist_rank.mean())

# ============================
# 4) 장르 rank (기존 유지)
# ============================
genre_rank = train.groupby('genre')['box_off_num'].mean().rank()
train['genre_rank'] = train['genre'].map(genre_rank)
test['genre_rank'] = test['genre'].map(genre_rank).fillna(genre_rank.mean())

# ============================
# 5) log version of count vars
# ============================
for col in ['time', 'dir_prev_num', 'num_staff', 'num_actor', 'distributor_median_rank']:
    train[col + '_log'] = np.log1p(train[col])
    test[col + '_log'] = np.log1p(test[col])

# ============================
# 삭제해도 되는 문자열 변수들 제거
# ============================
remove_cols = ['title', 'release_time', 'release_dt', 'director']
train.drop(columns=remove_cols, inplace=True)
test.drop(columns=remove_cols, inplace=True)


In [ ]:
y = train['box_off_num']
X = train.drop(['box_off_num'], axis=1)

# ============================
# 문자(Object) 컬럼 완전 제거
# ============================

obj_cols = train.select_dtypes(include=['object']).columns
print("문자 컬럼 제거됨:", obj_cols.tolist())

train.drop(columns=obj_cols, inplace=True)
test.drop(columns=obj_cols, inplace=True)

# ==========================================
# 🔥 전처리 마지막: 모든 NaN 숫자 대체
# ==========================================

# 1) train에서 숫자 컬럼 NaN → 평균으로 채움
num_cols_train = train.select_dtypes(include=[np.number]).columns
train[num_cols_train] = train[num_cols_train].fillna(train[num_cols_train].mean())

# 2) test에서는 train 평균으로 채움 (데이터 누출 방지)
num_cols_test = test.select_dtypes(include=[np.number]).columns
test[num_cols_test] = test[num_cols_test].fillna(train[num_cols_train].mean())


# 스케일링
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
test_scaled = scaler.transform(test)

train_y_log = np.log1p(y)

문자 컬럼 제거됨: []


In [ ]:
# Base models
lasso = Lasso(alpha=0.0003, random_state=42)
ENet = ElasticNet(alpha=0.0005, l1_ratio=0.9)
KRR = KernelRidge(alpha=0.6, kernel='rbf')

model_xgb = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8
)

model_lgb = lgb.LGBMRegressor(
    objective='regression',
    num_leaves=31,
    learning_rate=0.03,
    n_estimators=3000,
    max_bin=255,
    bagging_fraction=0.8,
    bagging_freq=5,
    feature_fraction=0.8,
    min_data_in_leaf=20,
    min_sum_hessian_in_leaf=5,
    random_state=42
)

model_hgb = HistGradientBoostingRegressor(max_depth=6, max_iter=600)
model_et  = ExtraTreesRegressor(n_estimators=500, max_depth=None, random_state=42)


In [ ]:
class StackingAveragedModels:
    def __init__(self, base_models, meta_model, n_folds=5):
        self.base_models = base_models
        self.meta_model = meta_model
        self.n_folds = n_folds

    def fit(self, X, y):
        self.base_models_ = [list() for x in self.base_models]
        self.meta_model_ = clone(self.meta_model)
        kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=42)

        out_of_fold_predictions = np.zeros((X.shape[0], len(self.base_models)))

        for i, model in enumerate(self.base_models):
            for train_idx, val_idx in kfold.split(X, y):
                instance = clone(model)
                self.base_models_[i].append(instance)
                instance.fit(X[train_idx], y[train_idx])
                y_pred = instance.predict(X[val_idx])
                out_of_fold_predictions[val_idx, i] = y_pred

        self.meta_model_.fit(out_of_fold_predictions, y)
        return self

    def predict(self, X):
        meta_features = np.column_stack([
            np.column_stack([model.predict(X) for model in models]).mean(axis=1)
            for models in self.base_models_ ])
        return self.meta_model_.predict(meta_features)


In [ ]:
stacked_averaged_models = StackingAveragedModels(
    base_models=(ENet, KRR, lasso),
    meta_model=lasso
)

# ---- 1) 각 모델 log 예측 ----
stacked_averaged_models.fit(X_scaled, train_y_log)
stacked_log = stacked_averaged_models.predict(test_scaled)

model_xgb.fit(X_scaled, train_y_log)
xgb_log = model_xgb.predict(test_scaled)

model_lgb.fit(X_scaled, train_y_log)
lgb_log = model_lgb.predict(test_scaled)

model_hgb.fit(X_scaled, train_y_log)
hgb_log = model_hgb.predict(test_scaled)

model_et.fit(X_scaled, train_y_log)
et_log = model_et.predict(test_scaled)

# ---- 2) LOG 공간에서 평균 ----
ensemble_log = (stacked_log + xgb_log + lgb_log + hgb_log + et_log) / 5

# ---- 3) expm1 (단 1회) ----
final_pred = np.expm1(ensemble_log)

submission['box_off_num'] = final_pred
submission.to_csv("bestmodel_PATCH.csv", index=False)
print("Saved bestmodel_PATCH.csv")


  File "c:\Users\jk102\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\jk102\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jk102\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jk102\anaconda3\Lib\subprocess.

[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_sum_hessian_in_leaf is set=5, min_child_weight=0.001 will be ignored. Current value: min_sum_hessian_in_leaf=5
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_sum_hessian_in_leaf is set=5, min_child_weight=0.001 will be ignored. Current value: min_sum_hessian_in_leaf=5
[LightGBM] 